In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
glass = pd.read_csv("Glass.csv")
y = glass['Type']
X = glass.drop('Type', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,
                                                    stratify=y)

## Linear SVM

In [ ]:
svm = SVC(kernel='linear', C=1, decision_function_shape='ovr')
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print( classification_report(y_test, y_pred) )

In [ ]:
print(accuracy_score(y_test, y_pred))

## OVO

In [ ]:
svm = SVC(kernel='linear', C=1, decision_function_shape='ovo')
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
print(classification_report(y_test, y_pred))

# # Decision Function

In [ ]:

decision_scores = svm.decision_function(X_test)

print(decision_scores)


## # Linear Kernel + C Tuning

In [ ]:
Cs = np.linspace(0.001, 5, 20)
dfs = ['ovo', 'ovr']
scores = []

for c in Cs:

    for d in dfs:

        svm = SVC(
            kernel='linear',
            C=c,
            decision_function_shape=d
        )

        svm.fit(X_train, y_train)

        y_pred = svm.predict(X_test)

        scores.append([
            c,
            d,
            accuracy_score(y_test, y_pred)
        ])

df_scores = pd.DataFrame(
    scores,
    columns=['C', 'Decision_Function', 'Accuracy']
)

print(df_scores.sort_values("Accuracy", ascending=False).head())

## Without Scaling

In [ ]:
Cs=np.linspace(0.001,5,20)
scores=[]
for c in Cs:
    svm=SVC(kernel='linear',C=c)
    svm.fit(X_train,y_train)
    y_pred = svm.predict(X_test)
    scores.append([c,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['C','score'])
df_scores.sort_values('score',ascending=False)

## RBF without Scaling

In [ ]:
cs = np.linspace(0.001, 5, 20)
gs = np.linspace(0.001, 5, 20)
scores=[]
for c in cs:
    for g in gs:
        svm=SVC(kernel='rbf',C=c,gamma=g)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        scores.append([c,g,accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores,columns=['C','gamma','score'])
df_scores.sort_values('score',ascending=False)

## # StandardScaler + Linear SVM

In [ ]:
scaler = StandardScaler()

Cs = np.linspace(0.001, 5, 20)

dfs = ['ovo', 'ovr']

scores = []

for c in tqdm(Cs):

    for d in dfs:

        svm = SVC(
            kernel='linear',
            C=c,
            decision_function_shape=d
        )

        pipe = Pipeline([
            ('Scaler', scaler),
            ('SVM', svm)
        ])

        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_test)

        scores.append([
            c,
            d,
            accuracy_score(y_test, y_pred)
        ])

df_scores = pd.DataFrame(
    scores,
    columns=['C', 'Decision_Function', 'Accuracy']
)

print(df_scores.sort_values("Accuracy", ascending=False).head())

## # StandardScaler + RBF Kernel

In [ ]:
scaler = StandardScaler()

Cs = np.linspace(0.001, 5, 20)

Gs = np.linspace(0.001, 5, 20)

dfs = ['ovo', 'ovr']

scores = []

for c in tqdm(Cs):

    for g in Gs:

        for d in dfs:

            svm = SVC(
                kernel='rbf',
                C=c,
                gamma=g,
                decision_function_shape=d
            )

            pipe = Pipeline([
                ('Scaler', scaler),
                ('SVM', svm)
            ])

            pipe.fit(X_train, y_train)

            y_pred = pipe.predict(X_test)

            scores.append([
                c,
                g,
                d,
                accuracy_score(y_test, y_pred)
            ])

df_scores = pd.DataFrame(
    scores,
    columns=[
        'C',
        'Gamma',
        'Decision_Function',
        'Accuracy'
    ]
)

print(df_scores.sort_values("Accuracy", ascending=False).head())